# Correcting Validity

Your task is to check the "productionStartYear" of the DBPedia autos datafile for valid values.
The following things should be done:
- check if the field "productionStartYear" contains a year
- check if the year is in range 1886-2014
- convert the value of the field to be just a year (not full datetime)
- the rest of the fields and values should stay the same
- if the value of the field is a valid year in the range as described above,
  write that line to the output_good file
- if the value of the field is not a valid year as described above, 
  write that line to the output_bad file
- discard rows (neither write to good nor bad) if the URI is not from dbpedia.org
- you should use the provided way of reading and writing data (DictReader and DictWriter)
  They will take care of dealing with the header.

You can write helper functions for checking the data and writing the files, but we will call only the 
'process_file' with 3 arguments (inputfile, output_good, output_bad).

In [14]:
import csv
import pprint
import re


INPUT_FILE = 'autos.csv'
OUTPUT_GOOD = 'result/autos-valid.csv'
OUTPUT_BAD = 'result/FIXME-autos.csv'

def is_valid_year(year_str):
    """
    Check if the productionStartYear is valid.
    It must be a valid year between 1886 and 2014.
    """
    try:
        # Extract only the year part (handle case where datetime might be included)
        year = int(re.match(r"(\d{4})", year_str).group(0))
        return 1886 <= year <= 2014
    except (ValueError, AttributeError):
        return False

def process_file(input_file, output_good, output_bad):

   with open(input_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        header = reader.fieldnames

        with open(output_good, 'w', newline='', encoding='utf-8') as good_file, \
             open(output_bad, 'w', newline='', encoding='utf-8') as bad_file:
            
            writer_good = csv.DictWriter(good_file, fieldnames=header)
            writer_bad = csv.DictWriter(bad_file, fieldnames=header)
            
            # Write headers
            writer_good.writeheader()
            writer_bad.writeheader()

            # Iterate through the rows and process each one
            for row in reader:
                production_start_year = row.get('productionStartYear', '')

                if is_valid_year(production_start_year):
                    # Write to valid file if the year is correct
                    row['productionStartYear'] = production_start_year[:4]  # Keep only the year part
                    writer_good.writerow(row)
                else:
                    # Write to invalid file otherwise
                    writer_bad.writerow(row)


def test():

    process_file(INPUT_FILE, OUTPUT_GOOD, OUTPUT_BAD)


if __name__ == "__main__":
    test()

# Profiling

#!/usr/bin/env python
#-*- coding: utf-8 -*-

In this problem set you work with cities infobox data, audit it, come up with a
cleaning idea and then clean it up. In the first exercise we want you to audit
the datatypes that can be found in some particular fields in the dataset.
The possible types of values can be:
- NoneType if the value is a string "NULL" or an empty string ""
- list, if the value starts with "{"
- int, if the value can be cast to int
- float, if the value can be cast to float, but CANNOT be cast to int.
   For example, '3.23e+07' should be considered a float because it can be cast
   as float but int('3.23e+07') will throw a ValueError
- 'str', for all other values

The audit_file function should return a dictionary containing fieldnames and a 
SET of the types that can be found in the field. e.g.
{"field1": set([type(float()), type(int()), type(str())]),
 "field2": set([type(str())]),
  ....
}
The type() function returns a type object describing the argument given to the 
function. You can also use examples of objects to create type objects, e.g.
type(1.1) for a float: see the test function below for examples.

Note that the first three rows (after the header row) in the cities.csv file
are not actual data points. The contents of these rows should not be included
when processing data types. Be sure to include functionality in your code to
skip over or detect these rows.


In [15]:
import codecs
import csv
import pprint

CITIES = 'cities.csv'

FIELDS = ["name", "timeZone_label", "utcOffset", "homepage", "governmentType_label",
          "isPartOf_label", "areaCode", "populationTotal", "elevation",
          "maximumElevation", "minimumElevation", "populationDensity",
          "wgs84_pos#lat", "wgs84_pos#long", "areaLand", "areaMetro", "areaUrban"]

def is_int(value):
    try:
        int(value)
        return True
    except ValueError:
        return False

def is_float(value):
    try:
        float(value)
        return not is_int(value)  # Vérifie que ce n'est pas un int
    except ValueError:
        return False

def audit_file(filename, fields):
    fieldtypes = {field: set() for field in fields}

    with codecs.open(filename, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)

        # Ignorer les trois premières lignes après l'en-tête
        for _ in range(3):
            next(reader)

        # Audit des types de données pour chaque champ
        for row in reader:
            for field in fields:
                value = row[field]
                
                if value == "" or value == "NULL":
                    fieldtypes[field].add(type(None))  # NoneType
                elif value.startswith("{"):
                    fieldtypes[field].add(type([]))  # list
                elif is_int(value):
                    fieldtypes[field].add(type(1))  # int
                elif is_float(value):
                    fieldtypes[field].add(type(1.1))  # float
                else:
                    fieldtypes[field].add(type(""))  # string

    return fieldtypes


def test():
    fieldtypes = audit_file(CITIES, FIELDS)

    pprint.pprint(fieldtypes)

    # Example of a test case for areaLand
    assert fieldtypes["areaLand"] == set([type(1.1), type([]), type(None)]) 
    
    
if __name__ == "__main__":
    test()


{'areaCode': {<class 'list'>, <class 'str'>, <class 'int'>, <class 'NoneType'>},
 'areaLand': {<class 'list'>, <class 'float'>, <class 'NoneType'>},
 'areaMetro': {<class 'list'>, <class 'float'>, <class 'NoneType'>},
 'areaUrban': {<class 'list'>, <class 'float'>, <class 'NoneType'>},
 'elevation': {<class 'list'>, <class 'float'>, <class 'NoneType'>},
 'governmentType_label': {<class 'list'>, <class 'str'>, <class 'NoneType'>},
 'homepage': {<class 'list'>, <class 'str'>, <class 'NoneType'>},
 'isPartOf_label': {<class 'NoneType'>, <class 'str'>, <class 'list'>},
 'maximumElevation': {<class 'list'>, <class 'float'>, <class 'NoneType'>},
 'minimumElevation': {<class 'float'>, <class 'NoneType'>},
 'name': {<class 'NoneType'>, <class 'str'>, <class 'list'>},
 'populationDensity': {<class 'list'>, <class 'float'>, <class 'NoneType'>},
 'populationTotal': {<class 'list'>, <class 'int'>, <class 'NoneType'>},
 'timeZone_label': {<class 'list'>, <class 'str'>, <class 'NoneType'>},
 'utcOff

# Crossfield Auditing

In this problem set you work with cities infobox data, audit it, come up with a
cleaning idea and then clean it up.

If you look at the full city data, you will notice that there are couple of
values that seem to provide the same information in different formats: "point"
seems to be the combination of "wgs84_pos#lat" and "wgs84_pos#long". However,
we do not know if that is the case and should check if they are equivalent.

Finish the function check_loc(). It will recieve 3 strings: first, the combined
value of "point" followed by the separate "wgs84_pos#" values. You have to
extract the lat and long values from the "point" argument and compare them to
the "wgs84_pos# values, returning True or False.

Note that you do not have to fix the values, only determine if they are
consistent. To fix them in this case you would need more information. Feel free
to discuss possible strategies for fixing this on the discussion forum.

Once you are done editig the code of the check_loc function, call the function process_file and examine the results.

In [18]:
import csv
import pprint

CITIES = 'cities.csv'

def check_loc(point, lat, longi):
    """
    Cette fonction prend la valeur combinée 'point' (lat long) et les valeurs séparées 'lat' et 'long'.
    Elle extrait les latitudes et longitudes de 'point', et les compare avec les valeurs séparées.
    """
    try:
        # Gérer plusieurs formats possibles pour la colonne 'point'
        if '|' in point:
            # Format avec plusieurs paires de coordonnées, on sélectionne la première paire
            point = point.split('|')[0].strip()
        
        if ',' in point:
            # Format avec virgule: "33.08, 75.28"
            point_lat, point_long = point.split(',')
        elif ' ' in point:
            # Format avec espace: "33.08 75.28"
            point_lat, point_long = point.split()
        else:
            raise ValueError("Format de point non pris en charge")
        
        # Nettoyer les valeurs en supprimant les espaces superflus
        point_lat, point_long = point_lat.strip(), point_long.strip()

        # Arrondir les valeurs extraites et les comparer avec celles données
        if round(float(point_lat), 4) == round(float(lat), 4) and round(float(point_long), 4) == round(float(longi), 4):
            return True
        else:
            return False
    except Exception as e:
        # En cas d'erreur, on retourne False
        print(f"Erreur lors de la vérification des données : {e}")
        return False

def process_file(filename):
    data = []
    with open(filename, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        
        # Skipping the extra metadata (premières 3 lignes)
        for i in range(3):
            next(reader)

        # Processer chaque ligne
        for line in reader:
            # Appeler la fonction check_loc pour vérifier la localisation
            result = check_loc(line["point"], line["wgs84_pos#lat"], line["wgs84_pos#long"])
            if not result:
                # Afficher les données inconsistantes
                print(f"{line['name']}: {line['point']} != {line['wgs84_pos#lat']} {line['wgs84_pos#long']}")
            data.append(line)

    return data


def test():
    # Quelques tests pour vérifier que check_loc fonctionne correctement
    assert check_loc("33.08 75.28", "33.08", "75.28") == True
    assert check_loc("44.57833333333333 -91.21833333333333", "44.5783", "-91.2183") == True
    assert check_loc("44.57833333333333 -91.21833333333333", "44.5783333333", "-91.2183333333") == True

    print("Tous les tests ont réussi !")

if __name__ == "__main__":
    test()


AssertionError: 